In [ ]:
# ---------------------------------------------------------------------
# prepare_chunks.ipynb
# ---------------------------------------------------------------------
# Split a large recoil spectra file into smaller chunk CSVs for FUSE simulation.
# Each line in the input file corresponds to one recoil spectrum (possibly empty).
# ---------------------------------------------------------------------

import os
from pathlib import Path

In [ ]:
# ---------------- CONFIG ----------------
BASE_DIR = Path.cwd().resolve().parents[0]  
DATA_DIR = BASE_DIR / "data"
RAW_DIR  = DATA_DIR / "raw"

FILE = "sbi_n300000_low_shm.csv"
DATASET_NAME = Path(FILE).stem                
INPUT_FILE = DATASET_NAME + ".csv"      # one spectrum per line
CHUNK_SIZE = 1500                       # spectra per chunk (300_000 / 1500 = 200 chunks)
# ----------------------------------------

DATASET_DIR = DATA_DIR / DATASET_NAME
CHUNKS_DIR = DATASET_DIR / "chunks"

os.makedirs(CHUNKS_DIR, exist_ok=True)

SCRATCH_BASE = Path("/scratch/midway3/nreus/dark_matter_sbi") / DATASET_NAME
TRASH_DIR = SCRATCH_BASE                   # keep naming consistent
CSV_IN    = SCRATCH_BASE / "csv_input"     # one input CSV per chunk
FUSE_OUT  = SCRATCH_BASE / "fuse_data"     # FUSE output per chunk

In [ ]:
def create_chunks(input_csv, chunk_size, chunks_dir):
    with open(input_csv, "r") as f:
        lines = f.read().splitlines()

    total = len(lines)
    print(f"[prepare_chunks] Found {total} spectra (including empty lines).")

    chunk_paths = []
    for i in range(0, total, chunk_size):
        chunk_lines = lines[i:i + chunk_size]
        chunk_idx = i // chunk_size
        chunk_path = chunks_dir / f"chunk_{chunk_idx:04d}.csv"
        chunk_path.write_text("\n".join(chunk_lines) + "\n")
        chunk_paths.append(chunk_path)

    print(f"[prepare_chunks] Created {len(chunk_paths)} chunks in {chunks_dir}")
    return chunk_paths
    

In [ ]:
if __name__ == "__main__":
    input_csv = RAW_DIR / FILE
    chunk_paths = create_chunks(input_csv, CHUNK_SIZE, CHUNKS_DIR)
    print(f"[prepare_chunks] Done. {len(chunk_paths)} chunks ready for simulation.")